# 03 - Privacy Demonstration and Policy Recommendation

### Table of contents

## 0. Setup and Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib
from scipy import stats
import json

# visual style
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)
pallete_colors = {'Male': '#E67E22', 'Female': '#9B59B6', 'Unknown': '#8c8c8c'}

In [2]:
# Data loading and exploration
df = pd.read_json("../data/raw/raw_credit_applications.json")

print(df.head())
print(df.info())

       _id                                     applicant_info  \
0  app_200  {'full_name': 'Jerry Smith', 'email': 'jerry.s...   
1  app_037  {'full_name': 'Brandon Walker', 'email': 'bran...   
2  app_215  {'full_name': 'Scott Moore', 'email': 'scott.m...   
3  app_024  {'full_name': 'Thomas Lee', 'email': 'thomas.l...   
4  app_184  {'full_name': 'Brian Rodriguez', 'email': 'bri...   

                                          financials  \
0  {'annual_income': 73000, 'credit_history_month...   
1  {'annual_income': 78000, 'credit_history_month...   
2  {'annual_income': 61000, 'credit_history_month...   
3  {'annual_income': 103000, 'credit_history_mont...   
4  {'annual_income': 57000, 'credit_history_month...   

                                   spending_behavior  \
0  [{'category': 'Shopping', 'amount': 480}, {'ca...   
1  [{'category': 'Rent', 'amount': 608}, {'catego...   
2              [{'category': 'Rent', 'amount': 109}]   
3           [{'category': 'Fitness', 'amount': 5

In [3]:
df

,_id,applicant_info,financials,spending_behavior,decision,processing_timestamp,loan_purpose,notes
0,app_200,"{'full_name': 'Jerry Smith', 'email': 'jerry.s...","{'annual_income': 73000, 'credit_history_month...","[{'category': 'Shopping', 'amount': 480}, {'ca...","{'loan_approved': False, 'rejection_reason': '...",2024-01-15T00:00:00Z,NaN,NaN
1,app_037,"{'full_name': 'Brandon Walker', 'email': 'bran...","{'annual_income': 78000, 'credit_history_month...","[{'category': 'Rent', 'amount': 608}, {'catego...","{'loan_approved': False, 'rejection_reason': '...",NaN,NaN,NaN
2,app_215,"{'full_name': 'Scott Moore', 'email': 'scott.m...","{'annual_income': 61000, 'credit_history_month...","[{'category': 'Rent', 'amount': 109}]","{'loan_approved': True, 'interest_rate': 3.7, ...",NaN,vacation,NaN
3,app_024,"{'full_name': 'Thomas Lee', 'email': 'thomas.l...","{'annual_income': 103000, 'credit_history_mont...","[{'category': 'Fitness', 'amount': 575}]","{'loan_approved': True, 'interest_rate': 4.3, ...",NaN,NaN,NaN
4,app_184,"{'full_name': 'Brian Rodriguez', 'email': 'bri...","{'annual_income': 57000, 'credit_history_month...","[{'category': 'Entertainment', 'amount': 463}]","{'loan_approved': False, 'rejection_reason': '...",2024-01-15T00:00:00Z,NaN,NaN
...,...,...,...,...,...,...,...,...
497,app_468,"{'full_name': 'Patrick Martinez', 'email': 'pa...","{'annual_income': 22000, 'credit_history_month...","[{'category': 'Transportation', 'amount': 701}]","{'loan_approved': False, 'rejection_reason': '...",NaN,NaN,NaN
498,app_192,"{'full_name': 'Dennis Lopez', 'email': 'dennis...","{'annual_income': 78000, 'credit_history_month...","[{'category': 'Healthcare', 'amount': 650}]","{'loan_approved': True, 'interest_rate': 6.0, ...",2024-01-15T00:00:00Z,NaN,NaN
499,app_234,"{'full_name': 'Samuel Hill', 'email': 'samuel....","{'annual_income': 96000, 'credit_history_month...","[{'category': 'Insurance', 'amount': 526}]","{'loan_approved': False, 'rejection_reason': '...",NaN,education,NaN
500,app_306,"{'full_name': 'Anna White', 'email': 'anna.whi...","{'annual_income': 106000, 'credit_history_mont...","[{'category': 'Insurance', 'amount': 490}]","{'loan_approved': True, 'interest_rate': 3.1, ...",NaN,NaN,NaN


## 1. Identification of Personal Identifiable Information (PII)

### 1.1 Direct Identifiers

Direct identifiers are variables that allow to immediate identication of an individual without the need to combine any other data.

This datast contains the following **Direct Identifiers** :

* **applicant_info.full_name**: This field contains the full legal name of each applicant. A full name directly identifies a specific individual.

* **applicant_info.email**: Email addresses are unique to each individual and allow for direct contact. 

* **applicant_info.ssn**: The Social Security Number is a unique national identifier, directly identifiying the person to whom it belongs to.

* **applicant_info.ip_address**: An IP address is assigned to a singular device connected to the internet. While it doesn't directly reveal the persons identity, it can be linked to a single individual through an internet provider, for example.

* **aplicatint_info._id**: Although an ID doesn't directly identify someone, it is unique to each individual, while not significant outside the system, it can be linked to a single indivdual.

### 1.2 Quasi-Identifiers

Quasi-identifiers cannot identify a specific individual on their own. However, when combined with other attributes, they can significantly narrow down the search and potentially lead to the identification of a single individual.

This dataset contains the following **Quasi-Identifiers**:

* **applicant_info.gender**: A person’s gender alone cannot reveal their identity. However, when combined with other attributes such as date of birth or zip code, it can substantially reduce the number of possible matches, especially in smaller datasets.

* **application_info.date_of_birth**: Many people share the same birthday, so this field alone does not identify an individual. However, when combined with other variables such as zip code and gender, it can drastically narrow down the number of possible individuals. In many cases, only one person may match a specific combination.

* **application_info.zip_code**: A zip code does not directly identify a person, but it narrows the search to a specific geographic location. When paired with attributes such as date of birth and gender, it can significantly increase the likelihood of identifying a single individual.

## 2. Uniqueness Analysis of Quasi-Identifiers

### 2.1 Checking Possible individual Identification Through Quasi-Identifier combinations

In [4]:
# Quasi-Identifiers columns

df["date_of_birth"] = df["applicant_info"].apply(lambda x: x.get("date_of_birth") if isinstance(x, dict) else None)
df["gender"]        = df["applicant_info"].apply(lambda x: x.get("gender") if isinstance(x, dict) else None)
df["zip_code"]      = df["applicant_info"].apply(lambda x: x.get("zip_code") if isinstance(x, dict) else None)

#### 2.1.1 Checking for single variables 

In [5]:

# date_of_birth alone
groups = df.groupby(["date_of_birth"]).size().reset_index(name="count")
print("date_of_birth unique:", (groups["count"] == 1).sum())

# zip_code alone
groups = df.groupby(["zip_code"]).size().reset_index(name="count")
print("zip_code unique:", (groups["count"] == 1).sum())

# gender alone
groups = df.groupby(["gender"]).size().reset_index(name="count")
print("gender unique:", (groups["count"] == 1).sum())

date_of_birth unique: 489
zip_code unique: 51
gender unique: 0


The analysis shows that date_of_birth alone uniquely identifies 489 out of 502 individuals. This means most applicants have a date of birth that appears only once in the dataset, which creates a high re identification risk.

The variable zip_code uniquely identifies 51 individuals. Although the risk is lower than for date_of_birth, it still allows a meaningful number of applicants to be singled out.

The variable gender does not uniquely identify any individual. This is expected because gender has a very small number of categories, so many applicants share the same value.

#### 2.1.2 Checking for combinations between 2 variables

In [6]:
# date_of_birth + zip_code
groups = df.groupby(["date_of_birth", "zip_code"]).size().reset_index(name="count")
print("dob + zip unique:", (groups["count"] == 1).sum())

# date_of_birth + gender
groups = df.groupby(["date_of_birth", "gender"]).size().reset_index(name="count")
print("dob + gender unique:", (groups["count"] == 1).sum())

# zip_code + gender
groups = df.groupby(["zip_code", "gender"]).size().reset_index(name="count")
print("zip + gender unique:", (groups["count"] == 1).sum())

dob + zip unique: 499
dob + gender unique: 495
zip + gender unique: 181


The combination of date_of_birth and zip_code singles out 499 out of 502 individuals. This means almost the entirety of the dataset can be uniquely identified when these two variables are used together.

The combination of date_of_birth and gender uniquely identifies 495 individuals. Again, date_of_birth drives most of the uniqueness, and the addition of gender further increases the level of singularity.

The combination of zip_code and gender uniquely identifies 181 individuals. Although significantly less than the other two combinations, it still uniquely identifies an important proportion of the dataset.

In [7]:
groups = df.groupby(["date_of_birth", "zip_code", "gender"]).size().reset_index(name="count")
print("dob + zip + gender unique:", (groups["count"] == 1).sum())

dob + zip + gender unique: 499


When we combine the three variables we can identify 499 people out of 502, representing almost the entire dataset. Previous data quality analysis identified duplicated records, which may explain the small number of non unique cases.

## 3. Pseudonymization and removal of Direct Identifiers

### 3.1. Hashing direct Identifiers

In [ ]:
df_before = df.copy()

def hash_value(value):
    if value is None:
        return None
    return hashlib.sha256(str(value).encode()).hexdigest()[:10]

df["applicant_info"] = df["applicant_info"].apply(
    lambda x: {**x,
               "full_name": hash_value(x.get("full_name")),
               "email": hash_value(x.get("email"))}
)

### 3.2. Removing direct identifiers

In [ ]:

df["applicant_info"] = df["applicant_info"].apply(
    lambda x: {k: v for k, v in x.items() if k not in ["ssn", "ip_address"]}
    )
df = df.drop(columns=["_id"])


### 3.3 Before and after Comparison

In [ ]:
example_before = df_before.iloc[0]
example_after = df.iloc[0]

comparison = pd.DataFrame({
    "name": [
        example_before["applicant_info"]["full_name"],
        example_after["applicant_info"]["full_name"]
    ],
    "email": [
        example_before["applicant_info"]["email"],
        example_after["applicant_info"]["email"]
    ],
    "ssn": [
        example_before["applicant_info"]["ssn"],
        None
    ],
    "ip_address": [
        example_before["applicant_info"]["ip_address"],
        None
    ],
    "_id": [
        example_before["_id"],
        None
    ]
}, index=["before", "after"])

comparison

,name,email,ssn,ip_address,_id
before,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,app_200
after,68ee17cf46,116648a776,None,None,None


## 4. GDPR Compliance Mapping

### 4.1 Lawful Basis

Under the GDPR, personal data must be processed under a valid lawful basis. This means an organization must have a legitimate reason for collecting and using personal data. NovaCred uses machine learning to make credit decisions, which requires access to applicants’ personal and financial information in order to train and operate its models.

In this context, the lawful basis for processing is the performance of a contract. Clients submit their personal and financial information when applying for a loan, and NovaCred processes this data to evaluate creditworthiness and decide whether the loan should be approved. Without processing this information, the company would not be able to assess applications or provide the requested financial service.



### 4.2  Data Minimization

The GDPR includes the principle of Data Minimization, which states that organizations should only collect and process the minimum amount of personal data necessary for their activities. If data is not required for their activity it should not be collected or stored.

In the case of NovaCred, the purpose of the data is to classify applicants as creditworthy or not. To check it, the company requires some financial and demographic information. However, several direct identifiers such as full name, email, IP address, and the internal system ID are not necessary for the functioning of the model and are still stored. For this reason, these fields were pseudonymized or removed to reduce the exposure of personal data.

### 4.3 Storage Limitation

The principle of Storage Limitation, states that data should only be stored for the purpose it was collected. Once it is no longer required, it should be deleted.

For NovaCred, personal and financial information, is stored to evaluate credit worthyness. Once a decision has been made regarding the approval or rejection of a loan, they no longer need to store all the initial data. For applicants, whose application was rejected, sotring their data won't be necessary after their process is closed. 

## 5. Privacy and Concern Gaps

## 6. Governance Recommendations